# Real-Dataset LSTM Training

Notebook ini menjalankan workflow deep learning untuk training LSTM pada dataset traffic asli. Seluruh bagian deep learning diletakkan di notebook agar eksperimen training, metrik, dan artifact bisa ditinjau secara interaktif.

Output utama notebook:
- sequence arrays train/validation/test,
- checkpoint model PyTorch `model.pt`,
- `training_history.json`, `model_config.json`, dan `metrics.json`,
- test metrics dalam skala terstandardisasi dan skala asli km/h,
- entry model aktif di local model registry.

## 1. Setup Project Path

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("C:/AIproject/AWAI")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
PROJECT_ROOT

WindowsPath('C:/AIproject/AWAI')

## 2. Imports and Configuration

In [2]:
from datetime import datetime
import json

import numpy as np
import pandas as pd
import torch

from traffic_prediction.config.settings import load_config
from traffic_prediction.data.processor import DataProcessor
from traffic_prediction.evaluation.metrics import calculate_regression_metrics
from traffic_prediction.features.offline import FeatureEngineer
from traffic_prediction.features.spatial import build_neighbor_mapping
from traffic_prediction.models.lstm import LSTMModelConfig, TrafficLSTM
from traffic_prediction.models.registry import ModelRegistry
from traffic_prediction.pipelines.offline import select_feature_columns
from traffic_prediction.training.trainer import LSTMTrainer, TrainingLoopConfig

config = load_config(project_root=PROJECT_ROOT)
run_stamp = datetime.now().strftime("lstm-real-%Y%m%d-%H%M%S")
report_dir = config.paths.reports_dir / run_stamp
report_dir.mkdir(parents=True, exist_ok=True)

print("Run:", run_stamp)
print("Traffic CSV:", config.paths.traffic_csv)
print("Roads CSV:", config.paths.roads_csv)
print("Reports:", report_dir)

Run: lstm-real-20260518-032721
Traffic CSV: C:\AIproject\AWAI\dataset\traffic_data_latest (1).csv
Roads CSV: C:\AIproject\AWAI\dataset\roads_master.csv
Reports: C:\AIproject\AWAI\artifacts\reports\lstm-real-20260518-032721


## 3. Load, Validate, Clean, and Engineer Features

In [3]:
processor = DataProcessor(config.data, config.features)

traffic_raw, validation_report = processor.load_traffic_csv(config.paths.traffic_csv)
roads = processor.load_roads_csv(config.paths.roads_csv)
cleaned = processor.clean(traffic_raw)

neighbor_mapping = build_neighbor_mapping(
    roads,
    neighbor_count=config.features.spatial_neighbor_count,
)
feature_engineer = FeatureEngineer(
    neighbor_mapping=neighbor_mapping,
    lag_steps=config.features.lag_steps,
    rolling_windows=config.features.rolling_windows,
)
featured = feature_engineer.extract_features(cleaned)
feature_columns = select_feature_columns(featured)
manifest = processor.build_feature_manifest(feature_columns)

print("Raw rows:", len(traffic_raw))
print("Cleaned rows:", len(cleaned))
print("Featured rows:", len(featured))
print("Road count:", roads["road_id"].nunique())
print("Feature count:", len(feature_columns))

Raw rows: 138000
Cleaned rows: 138050
Featured rows: 138050
Road count: 50
Feature count: 41


## 4. Chronological Split, Scaling, and Sequence Creation

In [4]:
train, validation, test, split_stats = processor.chronological_split(featured)

train_scaled = processor.fit_transform_train(train, feature_columns)
validation_scaled = processor.transform_eval(validation)
test_scaled = processor.transform_eval(test)

X_train, y_train, train_metadata = processor.create_sequences(train_scaled, feature_columns)
X_validation, y_validation, validation_metadata = processor.create_sequences(validation_scaled, feature_columns)
X_test, y_test, test_metadata = processor.create_sequences(test_scaled, feature_columns)

np.savez_compressed(
    report_dir / "lstm_sequences.npz",
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation,
    X_test=X_test,
    y_test=y_test,
)

sequence_summary = {
    "run_id": run_stamp,
    "feature_count": len(feature_columns),
    "feature_columns": feature_columns,
    "split_statistics": split_stats.__dict__,
    "sequence_shapes": {
        "X_train": list(X_train.shape),
        "y_train": list(y_train.shape),
        "X_validation": list(X_validation.shape),
        "y_validation": list(y_validation.shape),
        "X_test": list(X_test.shape),
        "y_test": list(y_test.shape),
    },
}
(report_dir / "sequence_summary.json").write_text(
    json.dumps(sequence_summary, indent=2, default=str),
    encoding="utf-8",
)

sequence_summary["sequence_shapes"]

{'X_train': [85650, 12, 41],
 'y_train': [85650, 4, 1],
 'X_validation': [28050, 12, 41],
 'y_validation': [28050, 4, 1],
 'X_test': [22100, 12, 41],
 'y_test': [22100, 4, 1]}

## 5. Train LSTM

`FULL_TRAINING = True` menjalankan konfigurasi training utama. Untuk smoke run cepat, ubah menjadi `False`.

In [5]:
FULL_TRAINING = True

model_config = LSTMModelConfig(
    input_size=X_train.shape[-1],
    prediction_horizon=y_train.shape[1],
    hidden_sizes=(64, 32),
    dense_units=16,
    dropout=0.3,
    recurrent_dropout=0.2,
    bidirectional=False,
)

training_config = TrainingLoopConfig(
    max_epochs=config.training.max_epochs if FULL_TRAINING else 3,
    batch_size=config.training.batch_size,
    learning_rate=config.training.learning_rate,
    early_stopping_patience=15 if FULL_TRAINING else 2,
    lr_plateau_patience=5,
    lr_plateau_factor=0.5,
    warmup_epochs=10 if FULL_TRAINING else 1,
    warmup_start_factor=0.1,
    gradient_clip_norm=1.0,
    weight_decay=0.0001,
    random_seed=config.training.random_seed,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

registry = ModelRegistry(config.paths.models_dir / "registry.json")
trainer = LSTMTrainer(model_config=model_config, training_config=training_config)

training_result = trainer.train(
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation,
    artifact_root=config.paths.models_dir,
    registry=registry,
    model_version=run_stamp,
    extra_metadata={
        "notebook": "notebooks/01_lstm_real_dataset_training.ipynb",
        "sequence_summary": str(report_dir / "sequence_summary.json"),
        "sequence_arrays": str(report_dir / "lstm_sequences.npz"),
        "feature_manifest": manifest.to_dict(),
    },
)

training_result

TrainingResult(model_version='lstm-real-20260518-032721', artifact_path='C:\\AIproject\\AWAI\\artifacts\\models\\lstm-real-20260518-032721', checkpoint_path='C:\\AIproject\\AWAI\\artifacts\\models\\lstm-real-20260518-032721\\model.pt', best_epoch=2, train_loss=0.641089062359134, validation_loss=0.43069454024501025, validation_mae=0.43222642694565067, validation_rmse=0.6562732207306631, parameter_count=40724, registry_entry=ModelRegistryEntry(model_version='lstm-real-20260518-032721', artifact_path='C:\\AIproject\\AWAI\\artifacts\\models\\lstm-real-20260518-032721', created_at='2026-05-18T03:38:36.484050', model_type='lstm', framework='pytorch', metrics={'train_loss': 0.641089062359134, 'validation_loss': 0.43069454024501025, 'validation_mae': 0.43222642694565067, 'validation_rmse': 0.6562732207306631}, config={'model_config': {'input_size': 41, 'prediction_horizon': 4, 'hidden_sizes': (64, 32), 'dense_units': 16, 'dropout': 0.3, 'recurrent_dropout': 0.2, 'bidirectional': False}, 'train

## 6. Evaluate Test Set

In [6]:
checkpoint = torch.load(training_result.checkpoint_path, map_location="cpu")
model = TrafficLSTM(model_config)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

def predict_in_batches(X: np.ndarray, batch_size: int = 2048) -> np.ndarray:
    predictions = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32)
            predictions.append(model(batch).numpy())
    return np.concatenate(predictions, axis=0)

test_predictions_scaled = predict_in_batches(X_test)
test_metrics_scaled = calculate_regression_metrics(y_test, test_predictions_scaled)

test_actual_kmh = processor.scaler_store.inverse_transform_speed(y_test.reshape(-1, 1)).reshape(y_test.shape)
test_predictions_kmh = processor.scaler_store.inverse_transform_speed(
    test_predictions_scaled.reshape(-1, 1)
).reshape(test_predictions_scaled.shape)
test_metrics_kmh = calculate_regression_metrics(test_actual_kmh, test_predictions_kmh)

evaluation_summary = {
    "model_version": training_result.model_version,
    "validation_metrics_scaled": {
        "loss": training_result.validation_loss,
        "mae": training_result.validation_mae,
        "rmse": training_result.validation_rmse,
    },
    "test_metrics_scaled": test_metrics_scaled.to_dict(),
    "test_metrics_kmh": test_metrics_kmh.to_dict(),
    "checkpoint_path": training_result.checkpoint_path,
    "artifact_path": training_result.artifact_path,
}
(report_dir / "lstm_test_evaluation.json").write_text(
    json.dumps(evaluation_summary, indent=2, default=str),
    encoding="utf-8",
)

evaluation_summary

{'model_version': 'lstm-real-20260518-032721',
 'validation_metrics_scaled': {'loss': 0.43069454024501025,
  'mae': 0.43222642694565067,
  'rmse': 0.6562732207306631},
 'test_metrics_scaled': {'mae': 0.42940093421516307,
  'rmse': 0.6526669586330787,
  'mape': 173.1737025714103,
  'r2': 0.5404653246275661,
  'sample_count': 88400},
 'test_metrics_kmh': {'mae': 3.119325978130237,
  'rmse': 4.741212343091945,
  'mape': 10.748332526333208,
  'r2': 0.5404653189167612,
  'sample_count': 88400},
 'checkpoint_path': 'C:\\AIproject\\AWAI\\artifacts\\models\\lstm-real-20260518-032721\\model.pt',
 'artifact_path': 'C:\\AIproject\\AWAI\\artifacts\\models\\lstm-real-20260518-032721'}

## 7. Active Registry Check

In [7]:
active_model = registry.get_active()
active_model

ModelRegistryEntry(model_version='lstm-real-20260518-032721', artifact_path='C:\\AIproject\\AWAI\\artifacts\\models\\lstm-real-20260518-032721', created_at='2026-05-18T03:38:36.484050', model_type='lstm', framework='pytorch', metrics={'train_loss': 0.641089062359134, 'validation_loss': 0.43069454024501025, 'validation_mae': 0.43222642694565067, 'validation_rmse': 0.6562732207306631}, config={'model_config': {'input_size': 41, 'prediction_horizon': 4, 'hidden_sizes': [64, 32], 'dense_units': 16, 'dropout': 0.3, 'recurrent_dropout': 0.2, 'bidirectional': False}, 'training_config': {'max_epochs': 100, 'batch_size': 64, 'learning_rate': 0.001, 'early_stopping_patience': 15, 'lr_plateau_patience': 5, 'lr_plateau_factor': 0.5, 'warmup_epochs': 10, 'warmup_start_factor': 0.1, 'gradient_clip_norm': 1.0, 'weight_decay': 0.0001, 'random_seed': 42, 'device': 'cpu'}}, tags=['trained-model'], is_active=True)